# 00b — Internal raw-data integrity and provenance checks

**Internal quality-control notebook** (not part of the public replication set).
Validates every preserved raw input against its recorded SHA-256 checksum,
documents the classification of each preserved input, and cross-checks the
rebuilt Billboard song-artist universe against the preserved derived aggregate
(`one_row_per_song.csv`) and the duplicate-key diagnostics.

**Inputs.** `data/raw/` (checksummed raw study inputs), `data/raw_checksums.csv`.

**Outputs.** `outputs/data/results/raw_checksum_report.csv`,
`outputs/data/results/raw_input_classification.csv`,
`outputs/data/results/billboard_key_diagnostics.csv`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw_checksums.csv").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from IPython.display import display

from cdr import paths, build, util
from cdr.util import SNAPSHOTS, SNAPSHOT_LABEL, check, show_and_save_table, show_and_save_figure

paths.ensure_output_dirs()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
RESULTS = paths.RESULTS

## Raw-input integrity

Every preserved raw input must match its recorded SHA-256 checksum before any construction step runs.


In [2]:
checksum_report = build.validate_raw_checksums()
show_and_save_table(checksum_report, RESULTS / "raw_checksum_report.csv")
check(checksum_report["status"].eq("OK").all(), "all raw inputs match their recorded SHA-256 checksums")


,filename,role,status
0,data/raw/2025_Data/Hot100_2025.csv,Billboard Hot 100 weekly chart base,OK
1,data/raw/2025_Data/data_bill_spoty_merged.csv,Spotify August 2025 popularity snapshot,OK
2,data/raw/2025_Data/data_completa_IMDb.csv,IMDb soundtrack song-film linkage from student...,OK
3,data/raw/2025_Data/data_completa_boxoffice.csv,Box Office Mojo movie metadata and box-office ...,OK
4,data/raw/2025_Data/datos_spotify_agosto_2022.csv,Spotify August 2022 popularity snapshot,OK
5,data/raw/2025_Data/datos_spotify_julio_2017.csv,Spotify July 2017 popularity snapshot,OK
6,data/raw/2025_Data/datos_spotify_octubre_2016.csv,Spotify October 2016 popularity snapshot,OK
7,data/raw/2025_Data/one_row_per_song.csv,Billboard song-artist aggregate derived from H...,OK
8,data/raw/2025_Data/songs_movies_processed.RData,Legacy preserved song-movie RData cache for re...,OK
9,data/raw/lastfm_spotify_2016_2017/Spoty_Final_...,Spotify January 2016 low-support audit snapshot,OK


PASS: all raw inputs match their recorded SHA-256 checksums


## Classification of preserved study inputs

Inputs are classified as **source-level raw** (as collected from the platform), **earliest
preserved study input** (the oldest preserved processed form when the original collection
files no longer exist), **reconstructable derived shortcut** (regenerated below and therefore
not used as a primary runtime input), or **legacy cache / excluded audit input** (retained for
lineage checks only).


In [3]:
classification = pd.DataFrame(
    [
        ("2025_Data/Hot100_2025.csv", "source-level raw input", "weekly Billboard Hot 100 chart entries; basis of the song-artist universe"),
        ("2025_Data/one_row_per_song.csv", "reconstructable derived shortcut", "song-artist aggregate; reconstructed exactly from Hot100_2025.csv below and not used as a primary runtime input"),
        ("2025_Data/datos_spotify_octubre_2016.csv", "earliest preserved study input", "Spotify popularity snapshot, October 2016"),
        ("2025_Data/datos_spotify_julio_2017.csv", "earliest preserved study input", "Spotify popularity snapshot, July 2017"),
        ("2025_Data/datos_spotify_agosto_2022.csv", "earliest preserved study input", "Spotify popularity snapshot, August 2022"),
        ("2025_Data/data_bill_spoty_merged.csv", "earliest preserved study input", "Spotify popularity snapshot, August 2025"),
        ("2025_Data/data_completa_IMDb.csv", "earliest preserved study input", "IMDb soundtrack song-film linkage (student thesis lineage)"),
        ("2025_Data/data_completa_boxoffice.csv", "earliest preserved study input", "Box Office Mojo movie metadata (student thesis lineage)"),
        ("2025_Data/songs_movies_processed.RData", "legacy cache (excluded from runtime)", "legacy song-movie cache retained only for lineage cross-checks"),
        ("lastfm_spotify_2016_2017/data_lastfm_Jul31_2017.csv", "earliest preserved study input", "Last.fm July 2017 listener snapshot (raw Listeners field)"),
        ("lastfm_spotify_2016_2017/all_data_billboard_30_Jul_2017.csv", "earliest preserved study input", "Billboard companion table preserved with the July 2017 collection"),
        ("lastfm_spotify_2016_2017/Spoty_Final_ENE_2016_2.RData", "legacy cache (excluded from runtime)", "legacy processed frame from an early Spotify collection; preserved with the raw data and not used in any analysis"),
        ("student_thesis_linkage/BSPO_jul17_aparece.csv", "earliest preserved study input", "July 2017 Billboard-Spotify-film linkage input used to reconstruct Last.fm analytical support"),
    ],
    columns=["raw_input", "classification", "role"],
)
show_and_save_table(classification, RESULTS / "raw_input_classification.csv")


,raw_input,classification,role
0,2025_Data/Hot100_2025.csv,source-level raw input,weekly Billboard Hot 100 chart entries; basis ...
1,2025_Data/one_row_per_song.csv,reconstructable derived shortcut,song-artist aggregate; reconstructed exactly f...
2,2025_Data/datos_spotify_octubre_2016.csv,earliest preserved study input,"Spotify popularity snapshot, October 2016"
3,2025_Data/datos_spotify_julio_2017.csv,earliest preserved study input,"Spotify popularity snapshot, July 2017"
4,2025_Data/datos_spotify_agosto_2022.csv,earliest preserved study input,"Spotify popularity snapshot, August 2022"
5,2025_Data/data_bill_spoty_merged.csv,earliest preserved study input,"Spotify popularity snapshot, August 2025"
6,2025_Data/data_completa_IMDb.csv,earliest preserved study input,IMDb soundtrack song-film linkage (student the...
7,2025_Data/data_completa_boxoffice.csv,earliest preserved study input,Box Office Mojo movie metadata (student thesis...
8,2025_Data/songs_movies_processed.RData,legacy cache (excluded from runtime),legacy song-movie cache retained only for line...
9,lastfm_spotify_2016_2017/data_lastfm_Jul31_201...,earliest preserved study input,Last.fm July 2017 listener snapshot (raw Liste...


,raw_input,classification,role
0,2025_Data/Hot100_2025.csv,source-level raw input,weekly Billboard Hot 100 chart entries; basis ...
1,2025_Data/one_row_per_song.csv,reconstructable derived shortcut,song-artist aggregate; reconstructed exactly f...
2,2025_Data/datos_spotify_octubre_2016.csv,earliest preserved study input,"Spotify popularity snapshot, October 2016"
3,2025_Data/datos_spotify_julio_2017.csv,earliest preserved study input,"Spotify popularity snapshot, July 2017"
4,2025_Data/datos_spotify_agosto_2022.csv,earliest preserved study input,"Spotify popularity snapshot, August 2022"
5,2025_Data/data_bill_spoty_merged.csv,earliest preserved study input,"Spotify popularity snapshot, August 2025"
6,2025_Data/data_completa_IMDb.csv,earliest preserved study input,IMDb soundtrack song-film linkage (student the...
7,2025_Data/data_completa_boxoffice.csv,earliest preserved study input,Box Office Mojo movie metadata (student thesis...
8,2025_Data/songs_movies_processed.RData,legacy cache (excluded from runtime),legacy song-movie cache retained only for line...
9,lastfm_spotify_2016_2017/data_lastfm_Jul31_201...,earliest preserved study input,Last.fm July 2017 listener snapshot (raw Liste...


## Cross-checks of the rebuilt Billboard universe

In [4]:
billboard = build.rebuild_billboard_from_hot100()
check(len(billboard) == 32124, "Billboard song-artist universe has 32,124 records")

PASS: Billboard song-artist universe has 32,124 records


### Cross-check against the preserved derived aggregate

Aggregation rules recovered from the weekly chart file: `number_of_weeks` is the longest
observed chart run (maximum `time_on_chart`; recharting instances restart the counter),
`maximum_peak_position` is the best weekly peak across all chart runs, and the provenance-only
`minimum_worst_position` records the running worst-position tracker at the song's debut chart
week. The analytically used fields (chart debut, weeks, peak position) must reproduce the
preserved `one_row_per_song.csv` exactly; the provenance-only fields (consecutive weeks,
debut-week worst position) are validated separately and are not used in any analysis. After
this check the rebuilt table is authoritative and the preserved aggregate is not used again at
runtime.


In [5]:
preserved = build.load_preserved_billboard()
analytical = ["song", "artist", "chart_debut", "number_of_weeks", "maximum_peak_position"]
provenance_only = ["maximum_consecutive_weeks", "minimum_worst_position"]
a = billboard[analytical + provenance_only].sort_values(["song", "artist"]).reset_index(drop=True)
b = preserved[analytical + provenance_only].sort_values(["song", "artist"]).reset_index(drop=True)
b["chart_debut"] = pd.to_datetime(b["chart_debut"])
check(a[analytical].equals(b[analytical]), "analytically used fields reproduce one_row_per_song.csv exactly (32,124 records)")
check(a[provenance_only].equals(b[provenance_only]), "provenance-only fields reproduce under the documented aggregation rules")


PASS: analytically used fields reproduce one_row_per_song.csv exactly (32,124 records)
PASS: provenance-only fields reproduce under the documented aggregation rules


## Duplicate-key diagnostics and record-preservation check

Song titles and artist strings are preserved exactly as charted. A handful of distinct
Billboard records (for example holiday songs recharting under near-identical titles by
different artists, or artist strings differing only in punctuation) would collapse if the
song/artist pair were reduced to a punctuation-free string. These records stay distinct in the
authoritative universe: an earlier over-aggressive re-normalization collapsed three film-linked
records and produced a spurious 4,680 film-linked count instead of the authoritative 4,683.


In [6]:
key_diag = build.billboard_key_diagnostics(billboard)
n_groups = key_diag["alnum_id"].nunique()
print(f"{len(key_diag)} distinct Billboard records across {n_groups} punctuation-free key groups would collapse under over-aggressive normalization:")
show_and_save_table(key_diag.drop(columns=["alnum_id"]), RESULTS / "billboard_key_diagnostics.csv", max_rows=20)
check(len(key_diag) > n_groups, "distinct Billboard records are preserved (no key-based collapsing in the authoritative universe)")


79 distinct Billboard records across 39 punctuation-free key groups would collapse under over-aggressive normalization:


,song,artist,chart_debut,number_of_weeks,maximum_peak_position
390,30 For 30,SZA With Kendrick Lamar,2025-01-04,29,10
391,30 For 30,SZAWithKendrick Lamar,2025-07-26,6,36
886,Act II: Date @ 8,4Batz Featuring Drake,2024-01-20,15,7
887,Act II: Date @ 8,4batz Featuring Drake,2024-05-11,4,47
1332,All The Way,BigXthaPlug FeaturingBailey Zimmerman,2025-07-26,6,27
1331,All The Way,BigXthaPlug Featuring Bailey Zimmerman,2025-04-19,14,4
1529,Amen,Shaboozey&Jelly Roll,2025-07-26,6,57
1528,Amen,Shaboozey & Jelly Roll,2025-05-10,11,56
837,APT.,ROSE & Bruno Mars,2024-11-02,38,3
838,APT.,ROSE &Bruno Mars,2025-07-26,6,35


PASS: distinct Billboard records are preserved (no key-based collapsing in the authoritative universe)
